In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    sum as _sum, to_date, col, row_number, lit,
    current_timestamp, when,row_number
)

In [0]:
orders_bronze = spark.table("de_workspace26.ecommerce_badal.orders")

orders_clean = (orders_bronze
  .dropDuplicates(["order_id"])
  .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
  .withColumn("revenue", col("quantity") * col("unit_price"))
)

In [0]:
print(f"After dedup: {orders_clean.count()} rows")
orders_clean.display()

After dedup: 200 rows


order_id,customer_id,product_id,order_date,quantity,unit_price,status,region,_ingested_at,_source_file,revenue
ORD0001,CUST04,PROD06,2024-03-29,5,1799.0,shipped,North,2026-04-08T05:13:20.923059Z,orders.csv,8995.0
ORD0002,CUST15,PROD09,2024-02-01,4,499.0,placed,East,2026-04-08T05:13:20.923059Z,orders.csv,1996.0
ORD0003,CUST20,PROD06,2024-05-27,2,1799.0,delivered,North,2026-04-08T05:13:20.923059Z,orders.csv,3598.0
ORD0004,CUST08,PROD05,2024-01-21,2,2199.0,delivered,West,2026-04-08T05:13:20.923059Z,orders.csv,4398.0
ORD0005,CUST09,PROD08,2024-06-11,3,699.0,shipped,East,2026-04-08T05:13:20.923059Z,orders.csv,2097.0
ORD0006,CUST07,PROD05,2024-06-28,1,2199.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,2199.0
ORD0007,CUST18,PROD04,2024-02-11,4,3499.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,13996.0
ORD0008,CUST11,PROD01,2024-02-28,1,49999.0,delivered,West,2026-04-08T05:13:20.923059Z,orders.csv,49999.0
ORD0009,CUST09,PROD02,2024-02-24,5,1499.0,delivered,East,2026-04-08T05:13:20.923059Z,orders.csv,7495.0
ORD0010,CUST07,PROD08,2024-04-11,4,699.0,shipped,South,2026-04-08T05:13:20.923059Z,orders.csv,2796.0


In [0]:
customer_window = (Window
  .partitionBy("customer_id")
  .orderBy("order_date")
  .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

In [0]:
orders_clean = orders_clean.withColumn(
    "cumulative_revenue", _sum("revenue").over(customer_window)
)

In [0]:
customers_df = spark.table("de_workspace26.ecommerce_badal.customers").select(
    "customer_id", "city", "loyalty_tier"
)
products_df = spark.table("de_workspace26.ecommerce_badal.products").select(
    "product_id", "product_name", "category"
)

orders_enriched = (orders_clean
  .join(customers_df, on="customer_id", how="left")
  .join(products_df,  on="product_id",  how="left")
)


In [0]:

print(f"Enriched orders: {orders_enriched.count()} rows")
orders_enriched.display()

Enriched orders: 200 rows


product_id,customer_id,order_id,order_date,quantity,unit_price,status,region,_ingested_at,_source_file,revenue,cumulative_revenue,city,loyalty_tier,product_name,category
PROD04,CUST01,ORD0098,2024-02-21,1,3499.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,3499.0,3499.0,Delhi,Gold,Wireless Earbuds,Electronics
PROD07,CUST01,ORD0173,2024-03-11,1,34999.0,delivered,North,2026-04-08T05:13:20.923059Z,orders.csv,34999.0,38498.0,Delhi,Gold,Smartphone,Electronics
PROD01,CUST01,ORD0164,2024-03-24,1,49999.0,shipped,East,2026-04-08T05:13:20.923059Z,orders.csv,49999.0,88497.0,Delhi,Gold,Laptop Pro,Electronics
PROD07,CUST01,ORD0105,2024-05-05,1,34999.0,delivered,East,2026-04-08T05:13:20.923059Z,orders.csv,34999.0,123496.0,Delhi,Gold,Smartphone,Electronics
PROD08,CUST01,ORD0046,2024-06-08,5,699.0,cancelled,North,2026-04-08T05:13:20.923059Z,orders.csv,3495.0,126991.0,Delhi,Gold,Yoga Mat,Sports
PROD02,CUST01,ORD0027,2024-06-30,1,1499.0,shipped,North,2026-04-08T05:13:20.923059Z,orders.csv,1499.0,128490.0,Delhi,Gold,Denim Jacket,Clothing
PROD09,CUST02,ORD0041,2024-01-21,2,499.0,placed,North,2026-04-08T05:13:20.923059Z,orders.csv,998.0,998.0,Bengaluru,Bronze,Python Programming Book,Books
PROD06,CUST02,ORD0193,2024-03-03,2,1799.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,3598.0,4596.0,Bengaluru,Bronze,Blender,Home
PROD09,CUST02,ORD0080,2024-03-04,1,499.0,delivered,West,2026-04-08T05:13:20.923059Z,orders.csv,499.0,5095.0,Bengaluru,Bronze,Python Programming Book,Books
PROD04,CUST02,ORD0088,2024-04-17,4,3499.0,cancelled,North,2026-04-08T05:13:20.923059Z,orders.csv,13996.0,19091.0,Bengaluru,Bronze,Wireless Earbuds,Electronics


In [0]:
# 3d. Write silver.orders partitioned by region
(orders_enriched.write
  .format("delta")
  .mode("overwrite")
  .partitionBy("region")
  .saveAsTable("de_workspace26.ecommerce_badal.silver_orders")
)
print(f"silver_orders written: {orders_enriched.count()} rows")
spark.table("de_workspace26.ecommerce_badal.silver_orders").display()

silver_orders written: 200 rows


product_id,customer_id,order_id,order_date,quantity,unit_price,status,region,_ingested_at,_source_file,revenue,cumulative_revenue,city,loyalty_tier,product_name,category
PROD04,CUST01,ORD0098,2024-02-21,1,3499.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,3499.0,3499.0,Delhi,Gold,Wireless Earbuds,Electronics
PROD06,CUST02,ORD0193,2024-03-03,2,1799.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,3598.0,4596.0,Bengaluru,Bronze,Blender,Home
PROD06,CUST03,ORD0026,2024-01-06,5,1799.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,8995.0,8995.0,Chennai,Bronze,Blender,Home
PROD08,CUST03,ORD0022,2024-01-18,5,699.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,3495.0,12490.0,Chennai,Bronze,Yoga Mat,Sports
PROD04,CUST03,ORD0036,2024-02-12,4,3499.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,13996.0,26486.0,Chennai,Bronze,Wireless Earbuds,Electronics
PROD09,CUST03,ORD0194,2024-02-23,5,499.0,shipped,South,2026-04-08T05:13:20.923059Z,orders.csv,2495.0,28981.0,Chennai,Bronze,Python Programming Book,Books
PROD04,CUST03,ORD0131,2024-03-08,5,3499.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,17495.0,46476.0,Chennai,Bronze,Wireless Earbuds,Electronics
PROD07,CUST03,ORD0053,2024-03-11,1,34999.0,placed,South,2026-04-08T05:13:20.923059Z,orders.csv,34999.0,81475.0,Chennai,Bronze,Smartphone,Electronics
PROD04,CUST03,ORD0112,2024-04-14,4,3499.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,13996.0,210965.0,Chennai,Bronze,Wireless Earbuds,Electronics
PROD05,CUST03,ORD0075,2024-05-11,5,2199.0,delivered,South,2026-04-08T05:13:20.923059Z,orders.csv,10995.0,224057.0,Chennai,Bronze,Running Shoes,Sports


In [0]:
# 3e. Enable Change Data Feed on silver_orders
spark.sql("""
  ALTER TABLE de_workspace26.ecommerce_badal.silver_orders
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("CDF enabled on silver_orders")

CDF enabled on silver_orders


In [0]:
spark.sql("""
  SELECT * FROM table_changes('de_workspace26.ecommerce_badal.silver_orders', 3)
  LIMIT 10
""").display()


product_id,customer_id,order_id,order_date,quantity,unit_price,status,region,_ingested_at,_source_file,revenue,cumulative_revenue,city,loyalty_tier,product_name,category,_change_type,_commit_version,_commit_timestamp
PROD04,CUST01,ORD0098,2024-02-21,1,3499.0,delivered,South,2026-04-08T10:47:23.134481Z,orders.csv,3499.0,3499.0,Delhi,Gold,Wireless Earbuds,Electronics,insert,6,2026-04-08T10:48:05Z
PROD06,CUST02,ORD0193,2024-03-03,2,1799.0,delivered,South,2026-04-08T10:47:23.134481Z,orders.csv,3598.0,4596.0,Bengaluru,Bronze,Blender,Home,insert,6,2026-04-08T10:48:05Z
PROD06,CUST03,ORD0026,2024-01-06,5,1799.0,delivered,South,2026-04-08T10:47:23.134481Z,orders.csv,8995.0,8995.0,Chennai,Bronze,Blender,Home,insert,6,2026-04-08T10:48:05Z
PROD08,CUST03,ORD0022,2024-01-18,5,699.0,delivered,South,2026-04-08T10:47:23.134481Z,orders.csv,3495.0,12490.0,Chennai,Bronze,Yoga Mat,Sports,insert,6,2026-04-08T10:48:05Z
PROD04,CUST03,ORD0036,2024-02-12,4,3499.0,delivered,South,2026-04-08T10:47:23.134481Z,orders.csv,13996.0,26486.0,Chennai,Bronze,Wireless Earbuds,Electronics,insert,6,2026-04-08T10:48:05Z
PROD09,CUST03,ORD0194,2024-02-23,5,499.0,shipped,South,2026-04-08T10:47:23.134481Z,orders.csv,2495.0,28981.0,Chennai,Bronze,Python Programming Book,Books,insert,6,2026-04-08T10:48:05Z
PROD04,CUST03,ORD0131,2024-03-08,5,3499.0,delivered,South,2026-04-08T10:47:23.134481Z,orders.csv,17495.0,46476.0,Chennai,Bronze,Wireless Earbuds,Electronics,insert,6,2026-04-08T10:48:05Z
PROD07,CUST03,ORD0053,2024-03-11,1,34999.0,placed,South,2026-04-08T10:47:23.134481Z,orders.csv,34999.0,81475.0,Chennai,Bronze,Smartphone,Electronics,insert,6,2026-04-08T10:48:05Z
PROD04,CUST03,ORD0112,2024-04-14,4,3499.0,delivered,South,2026-04-08T10:47:23.134481Z,orders.csv,13996.0,210965.0,Chennai,Bronze,Wireless Earbuds,Electronics,insert,6,2026-04-08T10:48:05Z
PROD05,CUST03,ORD0075,2024-05-11,5,2199.0,delivered,South,2026-04-08T10:47:23.134481Z,orders.csv,10995.0,224057.0,Chennai,Bronze,Running Shoes,Sports,insert,6,2026-04-08T10:48:05Z


In [0]:
%sql 
show partitions de_workspace26.ecommerce_badal.silver_orders;

region
South
East
West
North


In [0]:
spark.sql("""
  CREATE TABLE IF NOT EXISTS de_workspace26.ecommerce_badal.silver_customers (
    customer_id      STRING,
    name             STRING,
    email            STRING,
    city             STRING,
    loyalty_tier     STRING,
    signup_date      STRING,
    is_current       BOOLEAN,
    effective_start  TIMESTAMP,
    effective_end    TIMESTAMP
  )
  USING DELTA
""")


DataFrame[]

In [0]:
from delta.tables import DeltaTable

initial_customers = (spark.table("de_workspace26.ecommerce_badal.customers")
  .drop("_ingested_at", "_source_file")
  .withColumn("signup_date", col("signup_date").cast("string"))
  .withColumn("is_current",      lit(True))
  .withColumn("effective_start", current_timestamp())
  .withColumn("effective_end",   lit(None).cast("timestamp"))
)
initial_customers.write.format("delta").mode("overwrite").saveAsTable("de_workspace26.ecommerce_badal.silver_customers")
print("Initial silver_customers loaded")
spark.table("de_workspace26.ecommerce_badal.silver_customers").display()

Initial silver_customers loaded


customer_id,name,email,city,loyalty_tier,signup_date,is_current,effective_start,effective_end
CUST01,Alice Johnson,alice.johnson@email.com,Delhi,Gold,2020-08-16,true,2026-04-08T05:54:27.001671Z,null
CUST02,Bob Smith,bob.smith@email.com,Bengaluru,Bronze,2021-07-17,true,2026-04-08T05:54:27.001671Z,null
CUST03,Priya Sharma,priya.sharma@email.com,Chennai,Bronze,2021-04-02,true,2026-04-08T05:54:27.001671Z,null
CUST04,Raj Patel,raj.patel@email.com,Hyderabad,Bronze,2020-07-28,true,2026-04-08T05:54:27.001671Z,null
CUST05,Meena Nair,meena.nair@email.com,Pune,Gold,2023-01-21,true,2026-04-08T05:54:27.001671Z,null
CUST06,Arjun Mehta,arjun.mehta@email.com,Kolkata,Bronze,2023-04-24,true,2026-04-08T05:54:27.001671Z,null
CUST07,Sunita Rao,sunita.rao@email.com,Ahmedabad,Silver,2020-03-06,true,2026-04-08T05:54:27.001671Z,null
CUST08,Kiran Das,kiran.das@email.com,Jaipur,Bronze,2020-07-10,true,2026-04-08T05:54:27.001671Z,null
CUST09,Lakshmi Iyer,lakshmi.iyer@email.com,Surat,Bronze,2021-04-21,true,2026-04-08T05:54:27.001671Z,null
CUST10,Vijay Kumar,vijay.kumar@email.com,Mumbai,Gold,2023-05-17,true,2026-04-08T05:54:27.001671Z,null


In [0]:
updates_df = (spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .load('/Volumes/de_workspace26/ecommerce_badal/raw/customers_update.csv')
  .withColumn("signup_date", col("signup_date").cast("string"))
  .withColumn("is_current",      lit(True))
  .withColumn("effective_start", current_timestamp())
  .withColumn("effective_end",   lit(None).cast("timestamp"))
)

silver_customers = DeltaTable.forName(spark, "de_workspace26.ecommerce_badal.silver_customers")

In [0]:
changed = (updates_df.alias("u")
  .join(
      spark.table("de_workspace26.ecommerce_badal.silver_customers")
           .filter("is_current = false")
           .alias("e"),
      col("u.customer_id") == col("e.customer_id")
  )
  .select("u.*")
)

(changed.write
  .format("delta")
  .mode("append")
  .saveAsTable("de_workspace26.ecommerce_badal.silver_customers")
)

print("SCD Type 2 MERGE complete")
spark.table("de_workspace26.ecommerce_badal.silver_customers").orderBy("customer_id", "effective_start").display()

SCD Type 2 MERGE complete


customer_id,name,email,city,loyalty_tier,signup_date,is_current,effective_start,effective_end
CUST01,Alice Johnson,alice.johnson@email.com,Delhi,Gold,2020-08-16,true,2026-04-08T05:54:27.001671Z,null
CUST02,Bob Smith,bob.smith@email.com,Bengaluru,Bronze,2021-07-17,true,2026-04-08T05:54:27.001671Z,null
CUST03,Priya Sharma,priya.sharma@email.com,Chennai,Bronze,2021-04-02,true,2026-04-08T05:54:27.001671Z,null
CUST04,Raj Patel,raj.patel@email.com,Hyderabad,Bronze,2020-07-28,true,2026-04-08T05:54:27.001671Z,null
CUST05,Meena Nair,meena.nair@email.com,Pune,Gold,2023-01-21,true,2026-04-08T05:54:27.001671Z,null
CUST06,Arjun Mehta,arjun.mehta@email.com,Kolkata,Bronze,2023-04-24,true,2026-04-08T05:54:27.001671Z,null
CUST07,Sunita Rao,sunita.rao@email.com,Ahmedabad,Silver,2020-03-06,true,2026-04-08T05:54:27.001671Z,null
CUST08,Kiran Das,kiran.das@email.com,Jaipur,Bronze,2020-07-10,true,2026-04-08T05:54:27.001671Z,null
CUST09,Lakshmi Iyer,lakshmi.iyer@email.com,Surat,Bronze,2021-04-21,true,2026-04-08T05:54:27.001671Z,null
CUST10,Vijay Kumar,vijay.kumar@email.com,Mumbai,Gold,2023-05-17,true,2026-04-08T05:54:27.001671Z,null


In [0]:
spark.sql("OPTIMIZE de_workspace26.ecommerce_badal.silver_orders ZORDER BY (customer_id, order_date)")
print("OPTIMIZE + ZORDER completed on silver_orders")

OPTIMIZE + ZORDER completed on silver_orders
